# Análise de Processos com Desconto em Folha

Consulta `processo.dbo.vw_ia_informacao` filtrando processos que tenham:
- pelo menos uma informação da **CCD**; e
- pelo menos uma informação da **proc_exe** cujo `resumo` contenha `desconto ... folha`.

In [16]:
import pandas as pd
from sqlalchemy import text

from ccd.notebook import setup

ctx = setup(llm=False)
engine = ctx.engine

In [23]:
sql = text("""
SELECT DISTINCT
    v_pe.numero_processo,
    v_pe.ano_processo,
    v_pe.DataPublicacao  AS data_pe,
    v_pe.resumo          AS resumo_pe,
    v_ccd.DataPublicacao AS data_ccd,
    v_ccd.resumo         AS resumo_ccd
FROM processo.dbo.vw_ata_informacao v_pe
JOIN processo.dbo.vw_ata_informacao v_ccd
      ON v_pe.numero_processo = v_ccd.numero_processo
     AND v_pe.ano_processo   = v_ccd.ano_processo
WHERE v_pe.setor  = 'PROC_EXE'
  AND v_pe.resumo  LIKE :resumo_pe_pat
  AND v_ccd.setor = 'CCD'
  AND v_ccd.resumo LIKE :resumo_ccd_pat
  AND v_ccd.DataPublicacao > v_pe.DataPublicacao
ORDER BY v_pe.ano_processo, v_pe.numero_processo, v_ccd.DataPublicacao
""")

df = pd.read_sql(
    sql,
    engine,
    params={
        "resumo_pe_pat":  "%desconto%folha%",
        "resumo_ccd_pat": "%notifica%",
    },
)

In [24]:
df.groupby(["numero_processo", "ano_processo"]).count()

,,data_pe,resumo_pe,data_ccd,resumo_ccd
numero_processo,ano_processo,,,,
000414,2024,1,1,1,1
000479,2024,1,1,1,1
001459,2023,1,1,1,1
002033,2024,2,2,2,2
002035,2024,1,1,1,1
002057,2024,1,1,1,1
002059,2024,1,1,1,1
002061,2024,1,1,1,1
002568,2024,1,1,1,1


In [25]:
df[(df.numero_processo == "003636") & (df.ano_processo == "2017")]

,numero_processo,ano_processo,data_pe,resumo_pe,data_ccd,resumo_ccd
0,003636,2017,2024-02-01 11:56:22.420,Quota. Desconto em folha. Remessa para DAE.......,2025-05-16 13:21:13.450,Notificação para desconto em folha...
1,003636,2017,2024-02-01 11:56:22.420,Quota. Desconto em folha. Remessa para DAE.......,2025-05-16 13:38:49.413,Notificação para desconto em folha...


## Recorte: débitos de Nereu Batista Linhares

Lista os processos com débitos vinculados a Nereu (`GenPessoa.Nome LIKE '%nereu%linhares%'`) cuja `vw_ata_informacao` da CCD contenha notificação para desconto em folha.

In [17]:
sql_nereu = text("""
SELECT DISTINCT
    v_ccd.numero_processo,
    v_ccd.ano_processo,
    v_ccd.DataPublicacao AS data_ccd,
    v_ccd.resumo         AS resumo_ccd,
    gp.Nome              AS pessoa,
    gp.Documento         AS cpf,
    ed.IdDebito,
    ed.ValorAPagar,
    ed.CodigoStatusDivida
FROM processo.dbo.vw_ata_informacao v_ccd
JOIN processo.dbo.Processos p
      ON  p.numero_processo = v_ccd.numero_processo
      AND p.ano_processo    = v_ccd.ano_processo
JOIN processo.dbo.Exe_Debito ed
      ON  ed.IdProcessoExecucao = p.IdProcesso
JOIN processo.dbo.Exe_DebitoPessoa edp
      ON  edp.IDDebito = ed.IdDebito
JOIN processo.dbo.GenPessoa gp
      ON  gp.IdPessoa = edp.IDPessoa
WHERE v_ccd.setor = 'CCD'
AND gp.Documento = '13006444434'
ORDER BY v_ccd.ano_processo, v_ccd.numero_processo, v_ccd.DataPublicacao
""")

df_nereu = pd.read_sql(
    sql_nereu,
    engine,
    params={
        "resumo_ccd_pat": "%notifica%",
        "nome_pattern":   "%nereu%linhares%",
    },
)
df_nereu

,numero_processo,ano_processo,data_ccd,resumo_ccd,pessoa,cpf,IdDebito,ValorAPagar,CodigoStatusDivida
0,002613,0024,2025-10-02 11:55:04.057,Suspensão do Débito..............................,NEREU BATISTA LINHARES,13006444434,27535,None,20
1,002613,0024,2026-01-14 12:30:24.913,Envio a Cons. Relator(a) para deliberação acer...,NEREU BATISTA LINHARES,13006444434,27535,None,20
2,005310,2017,2025-10-15 11:54:30.427,Deliberação do Relator.,NEREU BATISTA LINHARES,13006444434,27256,None,1
3,012970,2017,2025-09-26 11:08:12.783,Baixa de Dívida,NEREU BATISTA LINHARES,13006444434,19752,None,15
4,012970,2017,2025-09-26 11:08:12.783,Baixa de Dívida,NEREU BATISTA LINHARES,13006444434,23314,None,15
...,...,...,...,...,...,...,...,...,...
3289,000480,2026,2026-04-13 11:51:30.960,Evento do Processo Original,NEREU BATISTA LINHARES,13006444434,28130,None,1
3290,000480,2026,2026-04-13 11:51:30.990,Evento do Processo Original,NEREU BATISTA LINHARES,13006444434,28130,None,1
3291,000480,2026,2026-04-13 11:51:55.510,Certidão de Inscrição no Cadastro Informativo ...,NEREU BATISTA LINHARES,13006444434,28130,None,1
3292,000480,2026,2026-04-13 11:55:28.007,Certidão de Inscrição no Cadastro Informativo ...,NEREU BATISTA LINHARES,13006444434,28130,None,1


In [19]:
import re

from ccd.processo import get_info_file_path
from ccd.pdf import extract_text_from_pdf

PADRAO_NOTIF_DF = re.compile(
    r"notifica\w*[\s\S]{0,80}?desconto\s+em\s+folha",
    flags=re.IGNORECASE,
)

_SQL_CCD_INFOS = text("""
SELECT concat(rtrim(inf.setor),'_',inf.numero_processo,'_',inf.ano_processo,'_',RIGHT(concat('0000',inf.ordem),4),'.pdf') as arquivo,
       ppe.SequencialProcessoEvento as evento,
       inf.setor
FROM processo.dbo.vw_ata_informacao inf
INNER JOIN processo.dbo.Pro_ProcessoEvento ppe
    ON inf.idinformacao = ppe.idinformacao
WHERE inf.numero_processo = :numero
  AND inf.ano_processo    = :ano
  AND inf.setor           = 'CCD'
""")


def eventos_notificacao_df(numero: str, ano: str) -> list:
    """SequencialProcessoEvento das informações da CCD cujo PDF menciona notificação para desconto em folha."""
    df_ccd = pd.read_sql(_SQL_CCD_INFOS, engine, params={"numero": numero, "ano": ano})
    if df_ccd.empty:
        return []
    df_ccd["caminho_arquivo"] = df_ccd.apply(get_info_file_path, axis=1)
    casa = df_ccd["caminho_arquivo"].apply(
        lambda p: bool(PADRAO_NOTIF_DF.search(extract_text_from_pdf(p)))
    )
    return df_ccd.loc[casa, "evento"].tolist()


resumo_nereu = (
    df_nereu.groupby(["numero_processo", "ano_processo", "pessoa", "cpf"])
            .agg(qtd_debitos=("IdDebito", "nunique"))
            .reset_index()
)
resumo_nereu["eventos_notificacao_df"] = resumo_nereu.apply(
    lambda r: eventos_notificacao_df(r["numero_processo"], r["ano_processo"]),
    axis=1,
)
resumo_nereu

Error reading \\10.24.0.6\tce$\Informacoes_PDF\CCD\CCD_000480_2026_0068.pdf: Cannot read an empty file


,numero_processo,ano_processo,pessoa,cpf,qtd_debitos,eventos_notificacao_df
0,000069,2025,NEREU BATISTA LINHARES,13006444434,1,[]
1,000088,2023,NEREU BATISTA LINHARES,13006444434,1,[99]
2,000092,2023,NEREU BATISTA LINHARES,13006444434,1,[]
3,000093,2023,NEREU BATISTA LINHARES,13006444434,1,[]
4,000094,2023,NEREU BATISTA LINHARES,13006444434,1,[]
...,...,...,...,...,...,...
428,004898,2024,NEREU BATISTA LINHARES,13006444434,1,[]
429,004901,2024,NEREU BATISTA LINHARES,13006444434,1,[]
430,004908,2024,NEREU BATISTA LINHARES,13006444434,1,[]
431,005310,2017,NEREU BATISTA LINHARES,13006444434,1,[]


## Processos com `Exe_DescontoFolha`, agrupados por pessoa

Pega todos os processos que têm registro em `Exe_DescontoFolha`, junta com débito + pessoa, e produz um dicionário `dataframes_por_pessoa` em que cada chave é o nome da pessoa e o valor é um DataFrame de processos com a coluna `eventos_notificacao_df` (eventos da CCD cujo PDF menciona notificação para desconto em folha).

In [ ]:
sql_descontos = text("""
SELECT DISTINCT
    p.numero_processo,
    p.ano_processo,
    gp.Nome      AS pessoa,
    gp.Documento AS cpf,
    ed.IdDebito
FROM processo.dbo.Exe_DescontoFolha edf
JOIN processo.dbo.Exe_Debito ed
      ON ed.IdDebito = edf.IdDebito
JOIN processo.dbo.Exe_DebitoPessoa edp
      ON edp.IDDebito = ed.IdDebito
JOIN processo.dbo.GenPessoa gp
      ON gp.IdPessoa = edp.IDPessoa
JOIN processo.dbo.Processos p
      ON p.IdProcesso = ed.IdProcessoExecucao
""")

df_descontos = pd.read_sql(sql_descontos, engine)
df_descontos

In [ ]:
def resumir_por_pessoa(df: pd.DataFrame) -> pd.DataFrame:
    out = (df.groupby(["numero_processo", "ano_processo", "pessoa", "cpf"])
             .agg(qtd_debitos=("IdDebito", "nunique"))
             .reset_index())
    out["eventos_notificacao_df"] = out.apply(
        lambda r: eventos_notificacao_df(r["numero_processo"], r["ano_processo"]),
        axis=1,
    )
    return out


dataframes_por_pessoa = {
    pessoa: resumir_por_pessoa(grupo)
    for pessoa, grupo in df_descontos.groupby("pessoa")
}
list(dataframes_por_pessoa)